In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, shutil, subprocess
from pathlib import Path
DRIVE_ROOT=Path('/content/drive/MyDrive'); PARENT_DIR=DRIVE_ROOT/'CALSHIFT_Research'
PROJECT_ROOT=PARENT_DIR/'calshift-research'; CRED_DIR=DRIVE_ROOT/'.gitcreds'
subprocess.run(['git','config','--global','user.name','Md Anas Biswas'],check=False)
subprocess.run(['git','config','--global','user.email','anasbiswas@gmail.com'],check=False)
subprocess.run(['git','config','--global','credential.helper','store'],check=False)
for fn,dest in [('.git-credentials','/root/.git-credentials'),('.gitconfig','/root/.gitconfig')]:
    for cand in (PARENT_DIR/fn, CRED_DIR/fn):
        if cand.exists(): shutil.copy(cand,dest); os.chmod(dest,0o600); break
os.chdir(PROJECT_ROOT); sys.path.insert(0,str(PROJECT_ROOT/'src'))
subprocess.run(['git','pull','--ff-only','--quiet'],check=False)
import importlib
if 'config' in sys.modules: importlib.reload(sys.modules['config'])
import config
import numpy as np, pandas as pd
print('ready:', os.getcwd())


Mounted at /content/drive
ready: /content/drive/MyDrive/CALSHIFT_Research/calshift-research


In [ ]:
# =============================================================================
# Cell 2 - load recorded set sizes for all three datasets, harmonise the column
# name (NSL uses mean_set_size; CIC/UGR use set_size). Focal per dataset; NSL at
# rung 0.80. K (max set size) per dataset: NSL/UGR 5 classes, CIC 2 classes.
# =============================================================================
ALPHAS=[config.ALPHA_PRIMARY]+config.ALPHA_SENSITIVITY
FOCAL={'nslkdd':'R2L','cicids2017':'DoS','ugr16':'nerisbotnet'}
KMAX ={'nslkdd':5,'cicids2017':2,'ugr16':5}

# NSL from long parquet (has set size at all alphas/rungs); rung 0.80
nl=pd.read_parquet(config.PROC_DIR/'coverage_long_nslkdd.parquet')
nl=nl[(nl['score']=='aps')&(nl['variant']=='mondrian')&(nl['feasible'])&(np.isclose(nl['rung'],0.80))].copy()
nl=nl.rename(columns={'mean_set_size':'set_size'})
nl['dataset']='nslkdd'
NSL=nl[['dataset','class','protocol','alpha','coverage','set_size']]

cc=pd.read_csv(config.REPORTS_DIR/'coverage_primary_cicids2017.csv'); cc['dataset']='cicids2017'
CIC=cc[['dataset','class','protocol','alpha','coverage','set_size']]
uu=pd.read_csv(config.REPORTS_DIR/'coverage_primary_ugr16.csv'); uu['dataset']='ugr16'
UGR=uu[['dataset','class','protocol','alpha','coverage','set_size']]

allcov=pd.concat([NSL,CIC,UGR],ignore_index=True)
allcov=allcov[allcov['class']!='__marginal__']   # per-class rows only
print('rows:',len(allcov),'| datasets:',sorted(allcov.dataset.unique()))
print('set_size present + finite:', bool(allcov['set_size'].notna().all()))


In [ ]:
# =============================================================================
# Cell 3 - EFFICIENCY TABLE at primary alpha: per dataset per protocol, the focal
# coverage and focal set size, plus the mean over feasible classes. The story:
# where SHC undercovers, its focal set size is SMALLER than TSC/REC - small sets
# bought by missing coverage, not by confidence. Where SHC holds (UGR neris),
# set sizes are comparable.
# =============================================================================
prim=allcov[np.isclose(allcov['alpha'],config.ALPHA_PRIMARY)]
rows=[]
for ds in ['nslkdd','cicids2017','ugr16']:
    d=prim[prim.dataset==ds]; foc=FOCAL[ds]
    for proto in ['REC','TSC','SHC']:
        dp=d[d.protocol==proto]
        fc=dp[dp['class']==foc]
        foc_cov=float(fc['coverage'].mean()); foc_sz=float(fc['set_size'].mean())
        mean_cov=float(dp['coverage'].mean()); mean_sz=float(dp['set_size'].mean())
        rows.append({'dataset':ds,'protocol':proto,'focal':foc,'Kmax':KMAX[ds],
                     'focal_coverage':round(foc_cov,4),'focal_set_size':round(foc_sz,4),
                     'mean_coverage':round(mean_cov,4),'mean_set_size':round(mean_sz,4)})
eff=pd.DataFrame(rows)
print('EFFICIENCY at alpha',config.ALPHA_PRIMARY,'(focal + mean-over-classes):')
print(eff.to_string(index=False))

# the tradeoff read: SHC focal set size vs TSC focal set size, per dataset
print('\nfocal set-size gap (SHC minus TSC) and coverage gap:')
piv=eff.pivot(index='dataset',columns='protocol',values=['focal_coverage','focal_set_size','mean_coverage','mean_set_size'])
for ds in ['nslkdd','cicids2017','ugr16']:
    fsz_s=piv.loc[ds,('focal_set_size','SHC')]; fsz_t=piv.loc[ds,('focal_set_size','TSC')]
    msz_s=piv.loc[ds,('mean_set_size','SHC')];  msz_t=piv.loc[ds,('mean_set_size','TSC')]
    fcv_s=piv.loc[ds,('focal_coverage','SHC')]; fcv_t=piv.loc[ds,('focal_coverage','TSC')]
    print(f'  {ds:11s} | focal cov SHC={fcv_s:.3f} vs TSC={fcv_t:.3f} | '
          f'focal set SHC={fsz_s:.3f} vs TSC={fsz_t:.3f} (diff {fsz_s-fsz_t:+.3f}) | '
          f'mean set SHC={msz_s:.3f} vs TSC={msz_t:.3f} (diff {msz_s-msz_t:+.3f})')


In [ ]:
# =============================================================================
# Cell 4 - set size across alpha (sanity: sets shrink as alpha grows), verdict,
# figure, commit.
# =============================================================================
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
across=[]
for ds in ['nslkdd','cicids2017','ugr16']:
    foc=FOCAL[ds]
    for a in ALPHAS:
        d=allcov[(allcov.dataset==ds)&(np.isclose(allcov['alpha'],a))&(allcov['class']==foc)]
        for proto in ['REC','TSC','SHC']:
            dp=d[d.protocol==proto]
            across.append({'dataset':ds,'focal':foc,'alpha':a,'protocol':proto,
                           'focal_coverage':round(float(dp['coverage'].mean()),4),
                           'focal_set_size':round(float(dp['set_size'].mean()),4)})
across=pd.DataFrame(across)

eff.to_csv(config.REPORTS_DIR/'efficiency_setsize.csv',index=False)
across.to_csv(config.REPORTS_DIR/'efficiency_setsize_by_alpha.csv',index=False)
# record the actual SHC-minus-TSC gaps as facts (focal and mean), per dataset
gaps={}
pw=eff.pivot(index='dataset',columns='protocol',values=['focal_coverage','focal_set_size','mean_coverage','mean_set_size'])
for ds in ['nslkdd','cicids2017','ugr16']:
    gaps[ds]={'focal_cov_SHC_minus_TSC':round(float(pw.loc[ds,('focal_coverage','SHC')]-pw.loc[ds,('focal_coverage','TSC')]),4),
              'focal_setsize_SHC_minus_TSC':round(float(pw.loc[ds,('focal_set_size','SHC')]-pw.loc[ds,('focal_set_size','TSC')]),4),
              'mean_cov_SHC_minus_TSC':round(float(pw.loc[ds,('mean_coverage','SHC')]-pw.loc[ds,('mean_coverage','TSC')]),4),
              'mean_setsize_SHC_minus_TSC':round(float(pw.loc[ds,('mean_set_size','SHC')]-pw.loc[ds,('mean_set_size','TSC')]),4)}
verdict={'analysis':'set-size / efficiency: coverage bought at what width',
   'primary_alpha':config.ALPHA_PRIMARY,
   'shc_minus_tsc_gaps':gaps,
   'reads':('TSC/REC achieve ~nominal coverage; their set width is the true cost of valid coverage under '
            'shift. SHC undercovers, so any smaller SHC sets are NOT efficiency but failure (a too-tight '
            'source quantile). Read the recorded gaps: where the SHC set-size gap is negative alongside a '
            'negative coverage gap, small sets coincide with missed coverage. Where SHC holds (UGR neris) '
            'gaps are ~0. Interpret focal and mean gaps from the printed numbers, not a presumed direction.'),
   'efficiency_table':eff.to_dict('records')}
(config.REPORTS_DIR/'efficiency_verdict.json').write_text(json.dumps(verdict,indent=2))

# figure: coverage vs set size at primary alpha, focal points, by protocol
fig,ax=plt.subplots(figsize=(6.6,4.8))
col={'REC':'tab:green','TSC':'tab:blue','SHC':'tab:red'}; mk={'nslkdd':'^','cicids2017':'o','ugr16':'s'}
for _,r in eff.iterrows():
    ax.scatter(r['focal_set_size'],r['focal_coverage'],c=col[r['protocol']],marker=mk[r['dataset']],s=70,
               edgecolor='k',linewidth=0.4)
    ax.annotate(f"{r['dataset'][:3]}-{r['protocol']}",(r['focal_set_size'],r['focal_coverage']),
                fontsize=6,xytext=(3,3),textcoords='offset points')
ax.axhline(1-config.ALPHA_PRIMARY,color='grey',ls='--',lw=0.8,label=f'nominal {1-config.ALPHA_PRIMARY}')
ax.set_xlabel('focal prediction-set size'); ax.set_ylabel('focal coverage')
ax.set_title('Coverage vs set size (focal, alpha=0.05): SHC small sets = undercoverage')
ax.legend(fontsize=8); fig.tight_layout(); fig.savefig(config.REPORTS_DIR/'efficiency_setsize.png',dpi=140)
print('figure saved'); print('\n',json.dumps(verdict['reads']))

def git(*a, show=True):
    r=subprocess.run(['git',*a],capture_output=True,text=True)
    if show and (r.stdout or r.stderr): print((r.stdout+r.stderr).strip())
    return r
for s,dd in [('/root/.git-credentials',PARENT_DIR/'.git-credentials'),('/root/.gitconfig',PARENT_DIR/'.gitconfig')]:
    if os.path.exists(s): shutil.copy(s,dd)
os.chdir(PROJECT_ROOT); git('add','-A',show=False)
if git('status','--porcelain',show=False).stdout.strip():
    git('commit','-m','nb25: set-size / efficiency - SHC small sets coincide with undercoverage; TSC/REC pay width for valid coverage')
    r=git('push','-u','origin','main')
    if r.returncode: print('PUSH FAILED. Commit is safe locally.')
else: print('nothing to commit')
print(git('log','--oneline','-3',show=False).stdout)
